In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.staging.rdm.example.com/'

idp_name_institutional_admin = None
idp_username_institutional_admin = None
idp_password_institutional_admin = None

institution_id = 5
other_institution_id = 2

institution_name = 'Test'

default_result_path = None
close_on_fail = False
transition_timeout = 60000

forbidden_permission_status = "403"
forbidden_permission_message = "You do not have permission."

In [ ]:
if idp_name_institutional_admin is None:
    idp_name_institutional_admin = input(prompt=f'IdP Name for institutional admin: ')
if idp_username_institutional_admin is None:
    idp_username_institutional_admin = input(prompt=f'Username for {idp_name_institutional_admin}')
if idp_password_institutional_admin is None:
    idp_password_institutional_admin = getpass(prompt=f'Password for {idp_username_institutional_admin}@{idp_name_institutional_admin}')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# 機関管理者のアクセス制御を確認する

- サブシステム名: 管理者
- ページ/アドオン: 管理者
- 機能分類: 認証/認可
- シナリオ名: 機関管理者のアクセス制御を確認する
- 用意するテストデータ:   
   ・GRDM 管理者URL   
   ・アカウント(機関管理者、所属機関が、NII Storage ではない機関ストレージを使用していること、そのような所属機関が1件だけであること)    
   ・機関のID（削除されていない機関）   
   ・他機関のID（削除されていない機関）

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGakunin RDM管理者のトップページを表示する

・`/account/login/` にリダイレクトされ、ログイン画面が表示されること   
・リダイレクト後のURLが「https://{admin_domain}/account/login/」であること

In [ ]:
expected_login_url = f"{admin_rdm_url.rstrip('/')}/account/login/"

async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

    login_url = page.url
    assert expected_login_url in login_url, (
        f"リダイレクト後のURLが期待値と異なります。"
        f" expected: {expected_login_url}, actual: {login_url}"
    )

await run_pw(_step)

## アカウント(機関管理者）を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。


In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_institutional_admin, idp_username_institutional_admin, idp_password_institutional_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「ログイン可否設定」メニューを選択する	

「ログイン可否設定」ページが表示されること   
※ 画面見出し: eduPersonEntitlement

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/institutions/entitlements/"]').click()

    await expect(page.locator('//h2[text() = "eduPersonEntitlement"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、`?institution_id=<他機関ID>`を「ログイン可否設定」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/institutions/entitlements/?institution_id={other_institution_id}")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオン利用制御」メニューを選択する	

「機関アドオン一覧」ページが表示されること   
※ 画面見出し: アドオン利用制御 (機関名)

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/addons/"]').click(timeout=transition_timeout)

    await expect(page.locator("h2")).to_have_text(f"アドオン利用制御 ({institution_name})",timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「機関アドオン一覧」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/addons/{other_institution_id}/")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「利用統計」メニューを選択する	

「機関統計詳細」ページが表示されること   
※ 画面見出し: 利用統計

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/statistics/"]').click(timeout=transition_timeout)

    await expect(page.locator('//h2[text() = "利用統計"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「機関統計詳細」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/{other_institution_id}/")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「RDMワークフロー」メニューを選択する	

「ワークフローエンジン一覧」ページが表示されること   
※ 画面見出し: ワークフローエンジン管理

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/rdm_workflow/"]').click(timeout=transition_timeout)

    await expect(page.locator('//h2[contains(., "ワークフローエンジン管理")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「ワークフローエンジン一覧」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/rdm_workflow/{other_institution_id}/")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「証跡管理」メニューを選択する	

「タイムスタンプ対象ノード一覧」ページが表示されること   
リダイレクト後のURLが「https://{admin_domain}/timestampadd/{institution_id}/nodes/」であること    
※ 画面見出し: 〈機関名〉のノードリスト

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/timestampadd/"]').click(timeout=transition_timeout)

    timestampadd_url = page.url
    expected_url = f"{admin_rdm_url.rstrip('/')}/timestampadd/{institution_id}/nodes/"
    assert timestampadd_url == expected_url, (
        f"リダイレクト後のURLが期待値と異なります。"
        f" expected: {expected_url}, actual: {timestampadd_url}"
    )

    await expect(page.locator(f'//h2[contains(., "{institution_name}のノードリスト")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「タイムスタンプ対象ノード一覧」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/timestampadd/{other_institution_id}/nodes/")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「機関ストレージのクォータ」メニューを選択する

「機関ストレージのクォータ制御」ページが表示されること   
※ 画面見出し: 機関ストレージのクォータ制御

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/institutional_storage_quota_control/"]').click(timeout=transition_timeout)

    await expect(page.locator('//h2[text() = "機関ストレージのクォータ制御"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「機関ストレージのクォータ制御」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/institutional_storage_quota_control/user_list_by_institution_id/{other_institution_id}/")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/addons」を直接入力する	

「機関アドオン一覧」ページへリダイレクトされること   
※ 画面見出し: アドオン利用制御 (機関名)


In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/addons")

    await expect(page.locator("h2")).to_have_text(f"アドオン利用制御 ({institution_name})",timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics」を直接入力する	

「機関統計詳細」ページへリダイレクトされること   
リダイレクト後のURLが「https://{admin_domain}/statistics/{institution_id}/」であること   
※ 画面見出し: 利用統計

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics")

    statistics_url = page.url
    expected_url = f"{admin_rdm_url.rstrip('/')}/statistics/{institution_id}/"
    assert statistics_url == expected_url, (
        f"リダイレクト後のURLが期待値と異なります。"
        f" expected: {expected_url}, actual: {statistics_url}"
    )

    await expect(page.locator('//h2[text() = "利用統計"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/rdm_workflow」を直接入力する	

「ワークフローエンジン一覧」ページへリダイレクトされること   
※ 画面見出し: ワークフローエンジン管理

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/rdm_workflow")

    await expect(page.locator('//h2[contains(., "ワークフローエンジン管理")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/timestampadd」を直接入力する	

「タイムスタンプ対象ノード一覧」ページへリダイレクトされること    
※ 画面見出し: 〈機関名〉のノードリスト

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/timestampadd")

    await expect(page.locator(f'//h2[contains(., "{institution_name}のノードリスト")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/project_limit_number/settings」を直接入力する

「プロジェクト制限数設定一覧」ページが表示されること   
※ 画面見出し: プロジェクト作成数設定一覧

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/project_limit_number/settings")

    await expect(page.locator(f'//h2[text() = "プロジェクト作成数設定一覧"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/keymanagement/{institution_id}」を直接入力する	

「ユーザーキー一覧」ページが表示されること   
※ 画面見出し: なし。Browser titleのみ「ユーザ検索結果」

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/keymanagement/{institution_id}")

    await expect(page.locator('//table//th[normalize-space(.)="GUID"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「ユーザーキー一覧」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/keymanagement/{other_institution_id}")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/timestampsettings/{institution_id}/nodes」を直接入力する	

「タイムスタンプ設定ノード一覧」ページが表示されること   
リダイレクト後のURLが「https://{admin_domain}/timestampsettings/{institution_id}/nodes/」であること   
※ 画面見出し: 〈機関名〉のノードリスト

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/timestampsettings/{institution_id}/nodes")

    timestampsettings_url = page.url
    expected_url = f"{admin_rdm_url.rstrip('/')}/timestampsettings/{institution_id}/nodes/"
    assert timestampsettings_url == expected_url, (
        f"リダイレクト後のURLが期待値と異なります。"
        f" expected: {expected_url}, actual: {timestampsettings_url}"
    )

    await expect(page.locator(f'//h2[contains(., "{institution_name}のノードリスト")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「タイムスタンプ設定ノード一覧」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/timestampsettings/{other_institution_id}/nodes")

    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data/storage_location」を直接入力する	

「エクスポートデータストレージロケーション」ページが表示されること   
※ 画面見出し: エクスポートデータ保存場所   
※ Export Dataは現在無効化されている機能であり、UI上からはアクセスできませんが、ブラウザのアドレスバーにURLを直接入力することでアクセスできます。 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/storage_location")
    
    await expect(page.locator(f'//h2[text() = "エクスポートデータ保存場所"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data/deleted」を直接入力する	

「エクスポートデータ管理-削除済み一覧」ページが表示されること   
※ 画面見出し: 削除済みエクスポートデータ一覧    
※ Export Dataは現在無効化されている機能であり、UI上からはアクセスできませんが、ブラウザのアドレスバーにURLを直接入力することでアクセスできます。

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/deleted")
    
    await expect(page.locator(f'//h2[text() = "削除済みエクスポートデータ一覧"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data」を直接入力する	

「エクスポートデータ管理」ページが表示されること   
※ 画面見出し: エクスポートデータ一覧    
※ Export Dataは現在無効化されている機能であり、UI上からはアクセスできませんが、ブラウザのアドレスバーにURLを直接入力することでアクセスできます。

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data")
    
    await expect(page.locator(f'//h2[text() = "エクスポートデータ一覧"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data/institutions/{institution_id}/storages」を直接入力する	

「機関エクスポートデータ管理」ページが表示されること   
※ 画面見出し: 機関ストレージ一覧    
※ Export Dataは現在無効化されている機能であり、UI上からはアクセスできませんが、ブラウザのアドレスバーにURLを直接入力することでアクセスできます。

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/institutions/{institution_id}/storages")
    
    await expect(page.locator(f'//h2[text() = "機関ストレージ一覧"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、他機関のIDを「機関エクスポートデータ管理」ページのURLに入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/institutions/{other_institution_id}/storages")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/institutional_storage_quota_control」を直接入力する	

「機関ストレージのクォータ制御」ページへリダイレクトされること   
リダイレクト後のURLが「https://{admin_domain}/institutional_storage_quota_control/user_list_by_institution_id/{institution_id}/」であること   
※ 画面見出し: 機関ストレージのクォータ制御

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/institutional_storage_quota_control")

    institutional_storage_quota_control_url = page.url
    expected_url = f"{admin_rdm_url.rstrip('/')}/institutional_storage_quota_control/user_list_by_institution_id/{institution_id}/"
    assert institutional_storage_quota_control_url == expected_url, (
        f"リダイレクト後のURLが期待値と異なります。"
        f" expected: {expected_url}, actual: {institutional_storage_quota_control_url}"
    )

    await expect(page.locator(f'//h2[text() = "機関ストレージのクォータ制御"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/loa」を直接入力する	

「要求する保証レベルの設定」ページが表示されること   
※ 画面見出し: 要求する保証レベルの設定

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/loa")
    
    await expect(page.locator(f'//h2[text() = "要求する保証レベルの設定"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、`?institution_id=<他機関ID>`を「要求する保証レベルの設定」ページのURLに入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/loa/?institution_id={other_institution_id}")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/nodes」を直接入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/nodes")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/nodes/flagged_spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/nodes/flagged_spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/nodes/known_spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/nodes/known_spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/nodes/known_ham」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/nodes/known_ham")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprints」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprints")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprints/flagged_spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprints/flagged_spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprints/known_spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprints/known_spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprints/known_ham」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprints/known_ham")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprints/withdrawal_requests」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprints/withdrawal_requests")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/user_identification_information/institutions」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/user_identification_information/institutions")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/users」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/users")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/users/flagged_spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/users/flagged_spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/users/known_spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/users/known_spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/users/known_ham」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/users/known_ham")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/spam」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/spam")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/nodes/registration_list」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/nodes/registration_list")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/nodes/stuck_registration_list」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/nodes/stuck_registration_list")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/metadata/erad」を直接入力する

ページにアクセスできないこと   
`/account/login/` にリダイレクトされ、ログイン画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/metadata/erad")
    
    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

    login_url = page.url
    assert expected_login_url in login_url, (
        f"リダイレクト後のURLが期待値と異なります。"
        f" expected: {expected_login_url}, actual: {login_url}"
    )

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/institutions」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/institutions")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/institutions/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/institutions/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/brands」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/brands")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/brands/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/brands/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprint_providers」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprint_providers")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprint_providers/whitelist」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprint_providers/whitelist")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/preprint_providers/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprint_providers/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/asset_files」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/asset_files")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/asset_files/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/asset_files/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/collection_providers」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/collection_providers")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/collection_providers/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/collection_providers/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/registration_providers」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/registration_providers")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/registration_providers/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/registration_providers/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/meetings」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/meetings")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/metrics」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/metrics")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/subjects」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/subjects")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/osf_groups/search」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/osf_groups/search")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/osf_groups」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/osf_groups")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/banners」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/banners")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/banners/create」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/banners/create")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/maintenance」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/maintenance")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/institutions/institution_list」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/institutions/institution_list")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/institutional_storage/institutions」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/institutional_storage/institutions")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/project_limit_number/templates」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/project_limit_number/templates")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/account/register」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/account/register")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/account/settings/desk」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/account/settings/desk")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/admin」を直接入力する

入口ページが表示され、「変更のためのパーミッションがありません。」と表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/admin")
    
    await expect(page.get_by_role("heading", name="サイト管理")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("変更のためのパーミッションがありません。")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/admin/osf/osfuser/」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/admin/osf/osfuser/")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data/storage_location/institutions」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/storage_location/institutions")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data/institutions」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/institutions")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/export_data/management/institutions」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/export_data/management/institutions")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/management」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/management")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc」を直接入力する

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc")
    
    await expect(page.get_by_role("heading", name=forbidden_permission_status)).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}